# qbank — Generador de preguntas de opción múltiple

`qbank` genera variantes de preguntas cuyas respuestas correctas se determinan automáticamente mediante lógica proposicional.

Este notebook recorre las principales funcionalidades. Ejecuta las celdas en orden con **Shift+Enter**.

In [ ]:
from qbank import *

## 1. Bloque básico: `ProblemaTipo`

Un `ProblemaTipo` es una lista de componentes:
- **Cadenas de texto**: aparecen en todas las variantes.
- **Listas de `Supuesto`**: hipótesis alternativas (se elige una por variante).
- **Listas de `Cuestion`**: ítems de respuesta (se elige uno por variante).

Cada `Supuesto` y `Cuestion` tiene:
- `enunciado`: el texto que aparece en la pregunta.
- `semantica`: una fórmula lógica que determina si es verdadero o falso.

**Operadores disponibles** (variables con `v('nombre')`):

| Operador | Significado |
|----------|-------------|
| `v('A')` | variable proposicional A |
| `-v('A')` | negación de A |
| `v('A') & v('B')` | A y B |
| `v('A') \| v('B')` | A o B |
| `v('A') >> v('B')` | si A entonces B |
| `True` / `False` | siempre verdadero / siempre falso |

In [ ]:
p = ProblemaTipo([
    "Dado que ",
    [
        Supuesto("$\\mathcal{A}$ es verdadero, ", v('A')),
        Supuesto("$\\mathcal{B}$ es verdadero, ", v('B')),
    ],
    "indique qué afirmación es correcta: ",
    [
        Cuestion("$\\mathcal{A}$ es verdadero.", v('A')),
        Cuestion("$\\mathcal{B}$ es verdadero.", v('B')),
    ],
])

for etiqueta, enunciado, cuestiones in p:
    print(f"── Variante {etiqueta} ──")
    print(f"   {enunciado}")
    for texto, correcto, activa, exp in cuestiones:
        print(f"   {'✓' if correcto else '✗'} {texto}")
    print()

## 2. Preguntas paramétricas con `setup`

El parámetro `setup` es una función que genera valores numéricos o simbólicos en cada variante. Los textos admiten `@variable` para interpolación.

In [ ]:
import random

def numeros():
    a = random.randint(1, 9)
    b = random.randint(1, 9)
    return {'a': a, 'b': b, 'suma': a + b}

p_param = ProblemaTipo(
    [
        "Sean $a = @a$ y $b = @b$. ",
        [
            Cuestion("$a + b = @suma$", True),
            Cuestion("$a + b > 10$", lambda ns: ns['a'] + ns['b'] > 10),
            Cuestion("$a = b$",       lambda ns: ns['a'] == ns['b']),
        ],
    ],
    setup=numeros,
)

import itertools
for etiqueta, enunciado, cuestiones in itertools.islice(p_param, 4):
    print(f"── Variante {etiqueta} ──  {enunciado}")
    for texto, correcto, activa, exp in cuestiones:
        print(f"   {'✓' if correcto else '✗'} {texto}")
    print()

## 3. Preguntas de verdadero/falso: `ProblemaVF`

`ProblemaVF` genera variantes tomando aleatoriamente un subconjunto de un banco de cuestiones fijas.

In [ ]:
enunciado_vf = "Indique cuáles de las siguientes afirmaciones son verdaderas:"

banco = [
    ("La derivada de $x^2$ es $2x$.",          True),
    ("La integral de $2x$ es $x^2 + C$.",       True),
    ("$\\sin^2(x) + \\cos^2(x) = 2$.",          False),
    ("El número $e$ es irracional.",             True),
    ("$\\ln(1) = 1$.",                           False),
    ("$\\sqrt{2}$ es irracional.",               True),
]

p_vf = ProblemaVF(enunciado_vf, banco, NumPreguntas=3)

import itertools
for etiqueta, enunciado, cuestiones in itertools.islice(p_vf, 3):
    print(f"── Variante {etiqueta} ──")
    for texto, correcto in cuestiones:
        print(f"   {'✓' if correcto else '✗'} {texto}")
    print()

## 4. Guardar y cargar en JSON

Los problemas se pueden guardar en ficheros JSON y recuperarlos después, sin perder ninguna información.

In [ ]:
from qbank import save_problema, load_problema
import os

os.makedirs("mis_problemas", exist_ok=True)

# Guardar
save_problema(p, "mis_problemas/ejemplo.json")
print("Guardado en mis_problemas/ejemplo.json")

# Cargar
p2 = load_problema("mis_problemas/ejemplo.json")
print("Cargado correctamente. Variantes:")
for etiqueta, enunciado, cuestiones in p2:
    print(f"  Variante {etiqueta}: {enunciado}")

## 5. Editor visual

`ProblemaTipoEditor` es un formulario interactivo para diseñar problemas sin escribir la lista de listas a mano.

- **Nombre**: identificador del problema.
- **Setup** (opcional): código Python que genera valores aleatorios.
- **Slots**: los componentes del problema (texto, supuesto, cuestion, alternativas).
- **▶ Preview**: muestra las variantes generadas.
- **💾 Guardar**: guarda en el fichero indicado.
- **⬇ Descargar**: descarga el JSON directamente al navegador.

In [ ]:
from qbank import ProblemaTipoEditor

editor = ProblemaTipoEditor()

> **Nota**: si el editor muestra texto en lugar del formulario (`VBox(children=...`), recarga la página y vuelve a ejecutar las celdas desde el principio.

## 6. Preguntas multi-parte

`ProblemaMultiParte` agrupa varias sub-preguntas bajo un enunciado común. Cada sub-pregunta tiene su propio texto introductorio y sus propias opciones (todas se muestran, marcadas como correctas o incorrectas).

Es el formato apropiado para las preguntas *multi-parte* de AMC y *cloze* de Moodle.

In [ ]:
p_multi = ProblemaMultiParte(
    componentes=[
        "Una desviación típica es un indicador ",
        [
            Supuesto("de dispersión. ",        v('Disp')),
            Supuesto("de tendencia central. ", -v('Disp')),
        ],
    ],
    subpreguntas=[
        SubPregunta("(en cuanto a su objetivo)",
            [Cuestion("de tendencia central", -v('Disp')),
             Cuestion("de dispersión",          v('Disp'))]),
        SubPregunta("(en cuanto a su sensibilidad)",
            [Cuestion("sensible a valores extremos",  v('Disp')),
             Cuestion("no muy sensible a extremos",  -v('Disp'))]),
    ]
)

for etiqueta, enunciado, subpreguntas in p_multi:
    print(f"── Variante {etiqueta} ──")
    print(f"   {enunciado}")
    for intro, cuestiones in subpreguntas:
        print(f"   {intro}")
        for texto, correcto, _, _ in cuestiones:
            print(f"     {'✓' if correcto else '✗'} {texto}")
    print()

## 7. Exportar a AMC (LaTeX)

Las funciones `AMC*` generan el código LaTeX para Auto Multiple Choice. Se incluyen en el documento AMC con `\input{fichero.tex}`.

In [ ]:
import os
os.makedirs("exportaciones", exist_ok=True)

# Pregunta estándar
with open("exportaciones/preguntas_amc.tex", "w") as f:
    for etiqueta, enunciado, cuestiones in p:
        f.write(AMC("MiCuestionario", etiqueta, enunciado, cuestiones))

with open("exportaciones/preguntas_amc.tex") as f:
    print(f.read())

In [ ]:
with open("exportaciones/preguntas_multi_amc.tex", "w") as f:
    for etiqueta, enunciado, subpreguntas in p_multi:
        f.write(AMC_multipart("Estadistica", etiqueta, enunciado, subpreguntas))

with open("exportaciones/preguntas_multi_amc.tex") as f:
    print(f.read())

## 8. Exportar a Moodle (vía LaTeX)

Las funciones `Quiz*` generan un fichero `.tex` que, al compilarlo con `xelatex`, produce un `.xml` importable en Moodle.

```bash
xelatex MiCuestionario.tex   # genera MiCuestionario.xml
```

Importar en Moodle: Banco de preguntas → Importar → Formato Moodle XML.

In [ ]:
QuizMoodleLastCh("MiCuestionario", "exportaciones/", p)
print("Generado exportaciones/MiCuestionario.tex")

with open("exportaciones/MiCuestionario.tex") as f:
    print(f.read()[:800], "...")

In [ ]:
QuizClozeMulti("Estadistica", "exportaciones/", p_multi)
print("Generado exportaciones/Estadistica.tex")

with open("exportaciones/Estadistica.tex") as f:
    print(f.read()[:800], "...")

## Siguiente paso

Consulta el **Manual completo** en `Manual.org` para más detalles sobre:
- Preguntas con `setup` paramétrico y bancos masivos
- Opciones de exportación AMC (multicolumna, «Ninguna de las anteriores»)
- Exportación a código Python editable (`save_problema_py`)
- Carga de bancos de múltiples problemas (`load_banco` / `save_banco`)